# Real vs AI Image Detector (EfficientNetB4) - Google Colab Training

This notebook trains a production-level **EfficientNetB4** model to detect AI-generated vs Real images.
- **Real Images**: Extracted from CIFAR-10 (diverse categories, no face bias).
- **AI Images**: Synthetic transformations + AI face generations.
- **Training**: 2-Phase fine-tuning strategy (Head training -> Full unfreeze) with `EarlyStopping` and `ReduceLROnPlateau`.

In [ ]:
# 1. Mount Google Drive (to save the final model permanently)
from google.colab import drive
drive.mount('/content/drive')

# Install required dependencies (TensorFlow is pre-installed in Colab)
!pip install pillow matplotlib scikit-learn

In [ ]:
# 2. Check for GPU (CRITICAL for 20k images)
import tensorflow as tf

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
if tf.config.list_physical_devices('GPU'):
    print("✅ GPU is enabled and ready!")
else:
    print("❌ WARNING: GPU is NOT enabled! Training will take 15+ hours.")
    print("Go to: Runtime -> Change runtime type -> Hardware accelerator -> Select 'T4 GPU'.")

In [ ]:
# 3. Download and Prepare the Dataset (~20,000 images)
import os
import ssl
import time
import shutil
import urllib.request
import numpy as np
from PIL import Image, ImageFilter, ImageEnhance
from tensorflow.keras.datasets import cifar10

BASE_DIR  = "/content"
REAL_DIR  = os.path.join(BASE_DIR, "dataset_v2", "real")
AI_DIR    = os.path.join(BASE_DIR, "dataset_v2", "ai")

MAX_REAL  = 10000
MAX_AI    = 10000
SAVE_SIZE = (128, 128)

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

def download_url(url, save_path, min_size=3000):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, context=ctx, timeout=15) as resp:
            data = resp.read()
            if len(data) < min_size: return False
            with open(save_path, "wb") as f: f.write(data)
            return True
    except Exception: return False

print("Cleaning old dataset_v2/...")
shutil.rmtree(os.path.join(BASE_DIR, "dataset_v2"), ignore_errors=True)
os.makedirs(REAL_DIR, exist_ok=True)
os.makedirs(AI_DIR, exist_ok=True)

# 1. Save CIFAR-10 real images
print("📥 Loading CIFAR-10 (REAL images)...")
(x_train, _), (x_test, _) = cifar10.load_data()
all_images = np.concatenate([x_train, x_test], axis=0)
np.random.seed(42)
indices = np.random.permutation(len(all_images))[:MAX_REAL]
count = 0
for idx in indices:
    img = Image.fromarray(all_images[idx]).resize(SAVE_SIZE, Image.LANCZOS)
    img.save(os.path.join(REAL_DIR, f"real_{count:05d}.jpg"), quality=95)
    count += 1
real_count = count
print(f"✅ Saved {real_count} REAL images.")

# 2. Save AI images (Web faces + Synthetic CIFAR-10)
print("📥 Gathering AI images...")
count = 0
for i in range(200): # Grab 200 real AI faces quickly
    if count >= MAX_AI: break
    path = os.path.join(AI_DIR, f"ai_face_{count:05d}.jpg")
    if download_url("https://thispersondoesnotexist.com/", path, 5000):
        try:
            img = Image.open(path).convert("RGB").resize(SAVE_SIZE, Image.LANCZOS)
            img.save(path, quality=95)
            count += 1
        except:
            os.remove(path)
    time.sleep(0.1)

print(f"   Grabbed {count} web AI faces. Generating remaining synthetic images...")
np.random.seed(99)
indices = np.random.permutation(len(all_images))
for idx in indices:
    if count >= MAX_AI: break
    img = Image.fromarray(all_images[idx]).resize(SAVE_SIZE, Image.LANCZOS)
    img = img.filter(ImageFilter.SHARPEN).filter(ImageFilter.SHARPEN)
    img = ImageEnhance.Color(img).enhance(1.5 + np.random.uniform(0, 1.0))
    img = img.filter(ImageFilter.SMOOTH)
    img = ImageEnhance.Brightness(img).enhance(0.9 + np.random.uniform(0, 0.3))
    img.save(os.path.join(AI_DIR, f"ai_synth_{count:05d}.jpg"), quality=90)
    count += 1

ai_count = count
print(f"✅ Saved {ai_count} AI images.")
print("🎯 Dataset Generation Complete!")

In [ ]:
# 4. Define and Train the EfficientNetB4 Model
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report

DATASET_DIR = "/content/dataset_v2"
MODEL_PATH  = "/content/drive/MyDrive/efficientnet_trained.h5"
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
EPOCHS_HEAD = 10
EPOCHS_FINE = 20

# Build Generators
datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.2,
    rotation_range=20,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.85, 1.15]
)

train_gen = datagen.flow_from_directory(
    DATASET_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", subset="training", shuffle=True
)

val_gen = datagen.flow_from_directory(
    DATASET_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode="binary", subset="validation", shuffle=False
)

# Build Model
base_model = EfficientNetB4(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(1, activation="sigmoid")(x)
model = Model(inputs=base_model.input, outputs=output)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
]

# ---------- PHASE 1: Head Training ----------
print("\n--- PHASE 1: Training Model Head (Base Frozen) ---")
model.compile(optimizer=Adam(learning_rate=1e-4), loss="binary_crossentropy", metrics=["accuracy"])
h1 = model.fit(train_gen, epochs=EPOCHS_HEAD, validation_data=val_gen, callbacks=callbacks)

# ---------- PHASE 2: Fine-Tuning ----------
print("\n--- PHASE 2: Fine-Tuning Top 30 Layers ---")
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=1e-5), loss="binary_crossentropy", metrics=["accuracy"])
h2 = model.fit(train_gen, epochs=EPOCHS_FINE, validation_data=val_gen, callbacks=callbacks)

# ---------- EVALUATION & SAVING ----------
val_loss, val_acc = model.evaluate(val_gen)
print(f"\n✅ Final Validation Accuracy: {val_acc * 100:.2f}%")

print("\n--- Classification Report ---")
val_gen.reset()
preds = model.predict(val_gen, verbose=0)
y_pred = (preds > 0.5).astype(int).flatten()
print(classification_report(val_gen.classes, y_pred, target_names=["Real", "AI Generated"], digits=4))

model.save(MODEL_PATH)
print(f"\n💾 Model successfully saved to Google Drive: {MODEL_PATH}")

In [ ]:
# 5. Direct Prediction Testing (Without Flask)
import numpy as np
from google.colab import files
from PIL import Image
from tensorflow.keras.models import load_model

print("Loading trained model from Google Drive...")
try:
    inference_model = load_model(MODEL_PATH)
    print("Model loaded successfully!")
except Exception as e:
    print("Error loading model:", e)

print("\n--- Upload an image to test ---")
uploaded = files.upload()

for filename in uploaded.keys():
    img = Image.open(filename).convert("RGB")
    plt.imshow(img)
    plt.axis('off')
    plt.show()
    
    # Preprocess
    img_resized = img.resize((224, 224))
    img_array = np.array(img_resized, dtype="float32") / 255.0
    img_array = np.expand_dims(img_array, axis=0) # shape (1, 224, 224, 3)
    
    # Predict
    pred = float(inference_model.predict(img_array, verbose=0)[0][0])
    
    # 0 = Real, 1 = AI (matches ImageDataGenerator indices)
    if pred >= 0.5:
        label = "🤖 AI Generated"
        conf = pred * 100
    else:
        label = "📸 Real Image"
        conf = (1 - pred) * 100
        
    print(f"\nPrediction: {label}")
    print(f"Confidence: {conf:.2f}%")
    print("-" * 40)